# Original LoRA scale/add epilogue — T4 untested research candidate

Fuses the half-rounded adapter scale and base add only. Native separate q/v dropout,
A/B GEMMs, FP16 boundaries, backward and the defensive PEFT clone remain intact.
Six local CPU-interpreter fixtures are **not** pinned-stack PEFT/CUDA/NF4 validation.
Own-kernel useful share >=5%, performance, peak and quality remain unmeasured.

This notebook runs only in an already-prepared authorized T4 environment. It does not
install/download, change the task/harness/tests, submit to a queue, or manufacture a baseline ID.
Missing source/packages/public assets fail closed. Reuse prepared public model/data only;
all measured processes, adapter state, outputs and compiler caches are fresh.


In [ ]:
import os, sys, pathlib, uuid, hashlib, json, subprocess
# Explicit opt-in is required for a future authorized cloud GPU run.
AUTHORIZE_GPU_EXECUTION = False
# Set environment variables before Run All if prepared paths differ; no downloads occur.
OFFICIAL_REPO = pathlib.Path(os.environ.get('SOUP_OFFICIAL_REPO', '/kaggle/working/soup-kernel-challenge'))
HF_CACHE_TEXT = os.environ.get('SOUP_HF_HUB_CACHE') or os.environ.get('HF_HUB_CACHE')
if not HF_CACHE_TEXT:
    raise RuntimeError('Set SOUP_HF_HUB_CACHE to the already-prepared dedicated public Hugging Face hub cache')
HF_CACHE = pathlib.Path(HF_CACHE_TEXT)
RUN_ROOT = pathlib.Path(os.environ.get('SOUP_RUN_ROOT', '/kaggle/working/soup-lora-research'))
BACKEND = os.environ.get('SOUP_BACKEND', 'GCP')
assert BACKEND in ('GCP', 'KAGGLE')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE = RUN_ROOT / ('prepared-lora-' + uuid.uuid4().hex[:12])
PACKAGE.mkdir()
print('Prepared package:', PACKAGE)


In [ ]:
PAYLOAD = {'solution/__init__.py': '"""Research candidate: original LoRA scale/add epilogue, target T4 UNTESTED."""\ndef apply():\n    from .patch import install\n    install()\n', 'solution/epilogue.py': '"""Original narrow LoRA forward epilogue; not a GEMM/dropout replacement.\n\nContract: native half adapter B output -> half-rounded multiply by 2 ->\nadd with half/float base. The defensive PEFT clone is deliberately retained.\n"""\nimport atexit\nimport collections\nimport json\n\nimport torch\nimport triton\nimport triton.language as tl\n\n_COUNTS = collections.Counter()\n\n\ndef note(event, **details):\n    _COUNTS[event] += 1\n    if _COUNTS[event] == 1:\n        print(\'SOUP_LORA_EPILOGUE \' + json.dumps({\'event\': event, **details}, sort_keys=True), flush=True)\n\n\n@atexit.register\ndef _summary():\n    if _COUNTS:\n        print(\'SOUP_LORA_EPILOGUE \' + json.dumps({\'event\': \'process_exit_counts\', \'counts\': dict(_COUNTS)}, sort_keys=True), flush=True)\n\n\n@triton.jit\ndef lora_half_scale_add(Base, Delta, Out, N: tl.constexpr, BLOCK: tl.constexpr):\n    i = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)\n    keep = i < N\n    b = tl.load(Base + i, keep, other=0).to(tl.float32)\n    d = tl.load(Delta + i, keep, other=0).to(tl.float32)\n    # Native `delta * 2.0` writes a half tensor BEFORE the add. In particular,\n    # half overflow may not be canceled by an opposite-sign float base value.\n    scaled = (d * 2.0).to(tl.float16).to(tl.float32)\n    tl.store(Out + i, b + scaled, keep)\n\n\nclass _ScaleAdd(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, base, delta):\n        ctx.base_dtype, ctx.delta_dtype = base.dtype, delta.dtype\n        out = torch.empty_like(base, dtype=torch.promote_types(base.dtype, delta.dtype))\n        note(\'launch_attempt\', kernel=\'lora_half_scale_add\', base_dtype=str(base.dtype),\n             delta_dtype=str(delta.dtype), shape=list(delta.shape))\n        lora_half_scale_add[(triton.cdiv(delta.numel(), 1024),)](\n            base, delta, out, delta.numel(), BLOCK=1024, num_warps=4,\n            enable_fp_fusion=False,\n        )\n        # This means host launch submission, not synchronized GPU completion.\n        note(\'launch_submitted\', kernel=\'lora_half_scale_add\')\n        return out\n\n    @staticmethod\n    def backward(ctx, grad):\n        # Native AddBackward casts to each input dtype first. Native MulBackward\n        # then multiplies the half delta adjoint by 2. Do not scale in float and\n        # cast only afterwards: GradScaler overflow/rounding can differ.\n        g_base = grad.to(ctx.base_dtype) if ctx.needs_input_grad[0] else None\n        g_delta = grad.to(ctx.delta_dtype) * 2.0 if ctx.needs_input_grad[1] else None\n        return g_base, g_delta\n\n\ndef scale_add(base, delta, scaling):\n    """Conservative runtime dispatch; fallback has the original two operations."""\n    okay = (\n        type(base) is torch.Tensor and type(delta) is torch.Tensor\n        and type(scaling) in (int, float) and scaling == 2.0\n        and base.is_cuda and delta.is_cuda and base.device == delta.device\n        and delta.dtype == torch.float16 and base.dtype in (torch.float16, torch.float32)\n        and base.shape == delta.shape and base.ndim in (2, 3)\n        and base.is_contiguous() and delta.is_contiguous()\n        and delta.numel() > 0 and delta.shape[-1] in (1536, 256)\n        and torch.cuda.get_device_capability(base.device) == (7, 5)\n        and not torch.jit.is_tracing() and not torch.jit.is_scripting()\n        and not torch.compiler.is_compiling()\n    )\n    if not okay:\n        note(\'native_epilogue_fallback\')\n        return base + delta * scaling\n    return _ScaleAdd.apply(base, delta)\n', 'solution/patch.py': '"""Surgical PEFT integration; vendor forward is inspected, never bundled/credited.\n\nOnly a pinned vanilla scale/add pair is substituted in memory. The original\nforward owns base NF4, clone, dropout, adapters, GEMMs, casts and dispatch.\n"""\nimport ast\nimport functools\nimport hashlib\nimport importlib.metadata\nimport inspect\nimport os\nimport textwrap\n\nimport torch\n\nfrom .epilogue import note, scale_add\n\nEXPECTED_BNB_SHA256 = \'96ed6fda63d7c67a4646faea8ba38483e8be8670059a3ac3dd740a99bdab6950\'\n_OWNER = None\n\n\ndef transform_forward(source):\n    """Transform exactly one recognized pair; reject unexpected source layouts."""\n    tree = ast.parse(textwrap.dedent(source))\n    function = tree.body[0]\n    if not isinstance(function, ast.FunctionDef) or function.name != \'forward\' or function.decorator_list:\n        raise RuntimeError(\'Unexpected PEFT forward definition\')\n    scaled, summed = [], []\n    for node in ast.walk(function):\n        if not isinstance(node, ast.Assign) or len(node.targets) != 1 or not isinstance(node.targets[0], ast.Name):\n            continue\n        name = node.targets[0].id\n        if name == \'output\' and ast.unparse(node.value) == \'lora_B(lora_A(dropout(x))) * scaling\':\n            scaled.append(node)\n        if name == \'result\' and ast.unparse(node.value) == \'result + output\':\n            summed.append(node)\n    if len(scaled) != 1 or len(summed) != 1:\n        raise RuntimeError(\'Pinned vanilla scale/add pair is not unique\')\n    scaled[0].value = scaled[0].value.left\n    summed[0].value = ast.Call(\n        func=ast.Name(id=\'_soup_original_scale_add\', ctx=ast.Load()),\n        args=[ast.Name(id=n, ctx=ast.Load()) for n in (\'result\', \'output\', \'scaling\')], keywords=[])\n    return ast.fix_missing_locations(tree)\n\n\ndef install():\n    global _OWNER\n    if _OWNER is not None:\n        cls, original, wrapped = _OWNER\n        if cls.forward is not wrapped:\n            raise RuntimeError(\'LoRA wrapper ownership changed\')\n        return\n    if os.environ.get(\'TRITON_INTERPRET\') not in (None, \'\', \'0\'):\n        raise RuntimeError(\'Interpreter mode is diagnostic-only, never a target solution run\')\n    if importlib.metadata.version(\'peft\') != \'0.20.0\':\n        raise RuntimeError(\'LoRA candidate requires exact PEFT 0.20.0\')\n    from peft.tuners.lora import bnb as peft_bnb\n    # Check only the installed library source, never model/data/results.\n    module_source = inspect.getsource(peft_bnb)\n    if hashlib.sha256(module_source.encode()).hexdigest() != EXPECTED_BNB_SHA256:\n        raise RuntimeError(\'Pinned PEFT bnb.py bytes differ; no patch installed\')\n    cls = peft_bnb.Linear4bit\n    original = cls.forward\n    if original.__module__ != peft_bnb.__name__ or original.__qualname__ != \'Linear4bit.forward\':\n        raise RuntimeError(\'PEFT forward already patched by another component\')\n    tree = transform_forward(inspect.getsource(original))\n    scope = dict(original.__globals__)\n    scope[\'_soup_original_scale_add\'] = scale_add\n    exec(compile(tree, original.__code__.co_filename, \'exec\'), scope)\n    transformed = scope[\'forward\']\n\n    @functools.wraps(original)\n    def wrapped(layer, x, *args, **kwargs):\n        # The AST removes scaling before the vendor\'s optional output cast.\n        # Activate ONLY under actual CUDA fp16 autocast, where that cast branch\n        # is absent. All non-AMP and unrelated modules keep the native forward.\n        enabled = (type(layer) is cls and type(x) is torch.Tensor and x.is_cuda\n                   and torch.is_autocast_enabled(\'cuda\')\n                   and torch.get_autocast_dtype(\'cuda\') == torch.float16\n                   and not torch.is_inference_mode_enabled()\n                   and not torch.compiler.is_compiling()\n                   and not torch.jit.is_tracing() and not torch.jit.is_scripting()\n                   and layer.training and torch.is_grad_enabled()\n                   and not layer.disable_adapters and not layer.merged\n                   and len(layer.active_adapters) == 1 and not layer.lora_variant)\n        if enabled:\n            adapter = layer.active_adapters[0]\n            enabled = (adapter in layer.lora_A and adapter in layer.lora_B\n                       and layer.r[adapter] == 16 and layer.lora_alpha[adapter] == 32\n                       and layer.scaling[adapter] == 2.0\n                       and layer.lora_A[adapter].weight.shape == (16, 1536)\n                       and layer.lora_B[adapter].weight.shape in ((1536, 16), (256, 16))\n                       and layer.lora_A[adapter].weight.dtype == torch.float32\n                       and layer.lora_B[adapter].weight.dtype == torch.float32\n                       and type(layer.lora_dropout[adapter]) is torch.nn.Dropout\n                       and layer.lora_dropout[adapter].p == 0.05\n                       and not layer.lora_dropout[adapter].inplace\n                       and not args and not kwargs)\n        if not enabled:\n            note(\'native_forward_fallback\')\n            return original(layer, x, *args, **kwargs)\n        return transformed(layer, x, *args, **kwargs)\n\n    cls.forward = wrapped\n    _OWNER = (cls, original, wrapped)\n    note(\'installed\', clone_retained=True, native_dropout_retained=True,\n         kernel=\'lora_half_scale_add\', own_share=\'UNMEASURED\')\n\n\ndef uninstall():\n    global _OWNER\n    if _OWNER is None:\n        return\n    cls, original, wrapped = _OWNER\n    if cls.forward is not wrapped:\n        raise RuntimeError(\'Refusing to overwrite a foreign PEFT wrapper\')\n    cls.forward = original\n    _OWNER = None\n', 'run_prepared.py': '#!/usr/bin/env python3\n"""Offline paired official harness runner. Preparing this file does not run a GPU.\n\nInput directories are pre-provisioned. No install/download/queue submission.\n"""\nimport argparse\nimport fcntl\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport tempfile\nimport uuid\n\nimport process_guard\n\nCOMMIT = \'c7c2fcd15972ce8b898d8d3ff732ee192c817ceb\'\nREVISION = \'989aa7980e4cf806f80c7fef2b1adb7bc71aa306\'\nPINS = {\'torch\':\'2.11.0+cu128\',\'triton\':\'3.6.0\',\'soup-cli\':\'0.77.0\',\'transformers\':\'5.16.1\',\n        \'peft\':\'0.20.0\',\'trl\':\'0.29.1\',\'bitsandbytes\':\'0.50.2\',\'accelerate\':\'1.14.0\',\'datasets\':\'4.8.5\'}\nDATA = {\'train.jsonl\':\'93d036d912bd81c337dc29893e13a76f36161d96f896ef4e02492e2f1a21a0dd\',\n        \'eval_public.jsonl\':\'b7f85b78ba3186dce4deab7cfb8bf8dcf7998e1517c5749ac0f5005886c6f70e\'}\nKERNEL = \'lora_half_scale_add\'\n\n\ndef sha(p):\n    h = hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\n\ndef save(p, obj):\n    Path(p).write_text(json.dumps(obj, indent=2, allow_nan=False) + \'\\n\')\n\n\ndef command(cmd, cwd=None, env=None):\n    with tempfile.TemporaryFile(mode=\'w+t\') as out, tempfile.TemporaryFile(mode=\'w+t\') as err:\n        code=process_guard.run(cmd,cwd=cwd,env=env,stdout=out,stderr=err)\n        out.seek(0);err.seek(0);stdout,stderr=out.read(),err.read()\n        if code:raise subprocess.CalledProcessError(code,cmd,stdout,stderr)\n        return stdout.strip()\n\n\ndef frozen_hashes(repo):\n    names = command([\'git\', \'-C\', str(repo), \'ls-tree\', \'-r\', \'--name-only\', COMMIT,\n                     \'--\', \'harness\', \'task\', \'tests\']).splitlines()\n    if not names: raise RuntimeError(\'Official frozen source tree missing\')\n    out = {}\n    for name in names:\n        p = repo / name\n        expected = subprocess.run([\'git\',\'-C\',str(repo),\'show\',f\'{COMMIT}:{name}\'], check=True, capture_output=True).stdout\n        if p.is_symlink() or not p.is_file() or p.read_bytes() != expected:\n            raise RuntimeError(\'Frozen official file mismatch: \' + name)\n        out[name] = hashlib.sha256(expected).hexdigest()\n    status = command([\'git\',\'-C\',str(repo),\'status\',\'--porcelain\',\'--untracked-files=all\',\'--\',\'harness\',\'task\',\'tests\'])\n    if status: raise RuntimeError(\'Dirty frozen official paths: \' + status)\n    return out\n\n\ndef verify_assets(repo, cache):\n    model = cache / \'models--Qwen--Qwen2.5-1.5B-Instruct\'\n    if (model/\'refs/main\').read_text().strip() != REVISION:\n        raise RuntimeError(\'Cached model main must be the pinned public revision\')\n    snapshot = model/\'snapshots\'/REVISION\n    if not snapshot.is_dir(): raise RuntimeError(\'Pinned model snapshot is absent; provision it outside measurement\')\n    files = {}\n    for p in sorted(snapshot.rglob(\'*\')):\n        if p.is_file():\n            target = p.resolve(strict=True)\n            if not target.is_relative_to(model.resolve()): raise RuntimeError(\'Model link escapes selected cache\')\n            files[str(p.relative_to(cache))] = sha(target)\n    if not any(n.endswith(\'.safetensors\') for n in files): raise RuntimeError(\'Cached safetensors weights missing\')\n    for name, expected in DATA.items():\n        if sha(repo/\'data\'/name) != expected: raise RuntimeError(\'Public data hash mismatch: \' + name)\n    return {\'model_id\':\'Qwen/Qwen2.5-1.5B-Instruct\',\'revision\':REVISION,\'files\':files,\'data_files\':DATA}\n\n\ndef assess(result, out):\n    """Independent fail-closed completeness gate; never assigns an admission/score."""\n    problems = []\n    runs = result.get(\'runs\', {})\n    shares = []\n    for side in (\'base\', \'solution\'):\n        group = runs.get(side, [])\n        if len(group) != 2: problems.append(f\'{side}: expected exactly two runs\')\n        for run in group:\n            m = run.get(\'metrics\') or {}\n            if run.get(\'exit_code\') != 0 or m.get(\'exit_code\') != 0 or m.get(\'steps\') != 64 or run.get(\'global_step\') != 64:\n                problems.append(f"{run.get(\'run\')}: partial/failed native training")\n            if not isinstance(run.get(\'wall_s\'), (int,float)) or run[\'wall_s\'] <= 0:\n                problems.append(\'Missing full native command wall time\')\n            if not isinstance(run.get(\'gpu_peak_mib\'), (int,float)) or run[\'gpu_peak_mib\'] <= 0:\n                problems.append(\'Missing external GPU peak\')\n            adapter = Path(run.get(\'adapter\') or \'/missing\')\n            if not all((adapter/f).is_file() and (adapter/f).stat().st_size > 0 for f in [\'adapter_model.safetensors\',\'adapter_config.json\']):\n                problems.append(f\'{side}: saved adapter missing\')\n            if side == \'solution\':\n                ms = (m.get(\'own_kernel_ms\') or {}).get(KERNEL)\n                sec = m.get(\'timed_loop_s\')\n                count = (m.get(\'triton_launches\') or {}).get(KERNEL, 0)\n                path = (m.get(\'triton_files\') or {}).get(KERNEL, \'\')\n                if not (count > 0 and path.startswith(\'solution/\') and isinstance(ms,(int,float)) and math.isfinite(ms) and ms > 0 and isinstance(sec,(int,float)) and math.isfinite(sec) and sec > 0 and m.get(\'timed_steps\') == 62 and m.get(\'triton_watch_error\') is None):\n                    problems.append(f\'{side}: actual original kernel activation/time missing\')\n                else: shares.append(ms/(1000*sec))\n        ev = (result.get(\'evals\') or {}).get(side, {})\n        loss = ev.get(\'mean_loss\')\n        if ev.get(\'exit_code\') != 0 or not isinstance(loss,(int,float)) or not math.isfinite(loss) or loss <= 0:\n            problems.append(f\'{side}: public evaluation incomplete\')\n    if result.get(\'packages_changed\') != []: problems.append(\'Packages changed or not recorded\')\n    complete = not problems\n    return {\'status\': \'COMPLETE_MEASUREMENT\' if complete else \'FAILED_OR_INCOMPLETE\',\n            \'problems\':problems, \'score\':None, \'admission\':\'UNKNOWN\',\n            \'quality_scope\':\'public_only\', \'same_environment_baseline\':complete,\n            \'own_share_steps_3_onward\':shares if complete else None,\n            \'own_share_min\':min(shares) if complete and len(shares)==2 else None,\n            \'own_share_gate\':(min(shares)>=.05) if complete and len(shares)==2 else None,\n            \'ratios\':result.get(\'summary\',{}).get(\'ratios\') if complete else None,\n            \'raw_result_sha256\':sha(out/\'result.json\')}\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--repo\',type=Path,required=True)\n    p.add_argument(\'--cache\',type=Path,required=True)\n    p.add_argument(\'--solution\',type=Path,default=Path(__file__).parent/\'solution\')\n    p.add_argument(\'--out-root\',type=Path,required=True)\n    p.add_argument(\'--backend\',choices=[\'GCP\',\'KAGGLE\'],default=\'GCP\')\n    a=p.parse_args()\n    process_guard.prepare()\n    process_guard.install_signal_handlers()\n    if os.environ.get(\'TRITON_INTERPRET\') not in (None,\'\',\'0\'):\n        raise RuntimeError(\'TRITON_INTERPRET must be unset for target measurement\')\n    if platform.python_version() != \'3.13.15\': raise RuntimeError(\'Exact prepared Python 3.13.15 required; no install is performed\')\n    versions = {n:importlib.metadata.version(n) for n in PINS}\n    if versions != PINS: raise RuntimeError(\'Exact package pins mismatch: \'+json.dumps(versions))\n    if command([\'git\',\'-C\',str(a.repo),\'rev-parse\',\'HEAD\']) != COMMIT: raise RuntimeError(\'Official commit mismatch\')\n    frozen = frozen_hashes(a.repo)\n    assets = verify_assets(a.repo,a.cache)\n    os.environ.update(CUDA_VISIBLE_DEVICES=\'0\', CUDA_DEVICE_ORDER=\'PCI_BUS_ID\',\n                      PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\', HF_HUB_OFFLINE=\'1\',\n                      TRANSFORMERS_OFFLINE=\'1\', HF_DATASETS_OFFLINE=\'1\',\n                      HF_HUB_CACHE=str(a.cache.resolve()), PYTHONUNBUFFERED=\'1\')\n    # Shared across all six local notebooks. Keep fd live until process exit,\n    # including every baseline/candidate/evaluation/check. No polling/auto retry.\n    process_guard.check_quarantine(\'/tmp/soup-gpu0.lock.blocked\')\n    lock_fd = os.open(\'/tmp/soup-gpu0.lock\', os.O_CREAT | os.O_RDWR | os.O_NOFOLLOW, 0o600)\n    try:\n        fcntl.flock(lock_fd, fcntl.LOCK_EX | fcntl.LOCK_NB)\n    except BlockingIOError:\n        raise RuntimeError(\'GPU 0 is owned by another notebook; do not overlap jobs\')\n    process_guard.check_quarantine(\'/tmp/soup-gpu0.lock.blocked\')\n    process_guard.begin_quarantine(\'/tmp/soup-gpu0.lock.blocked\')\n    # A short-lived child may initialize CUDA; this parent must never hold a\n    # CUDA context across external-memory measurement of the training children.\n    probe_code = ("import json, torch; n=torch.cuda.device_count(); "\n                  "print(json.dumps({\'cuda\':torch.version.cuda,\'count\':n,"\n                  "\'names\':[torch.cuda.get_device_name(i) for i in range(n)]}))")\n    gpu_probe = json.loads(command([sys.executable, \'-c\', probe_code], env=dict(os.environ)))\n    if gpu_probe != {\'cuda\':\'12.8\',\'count\':1,\'names\':[\'Tesla T4\']}:\n        raise RuntimeError(\'Require one visible Tesla T4 and Torch compiled CUDA 12.8\')\n    available = sorted(os.sched_getaffinity(0))\n    if len(available)<4: raise RuntimeError(\'Four available logical CPUs required\')\n    os.sched_setaffinity(0, available[:4])\n    active = command([\'nvidia-smi\',\'--query-compute-apps=pid\',\'--format=csv,noheader,nounits\'])\n    if any(line.strip().isdigit() for line in active.splitlines()):\n        raise RuntimeError(\'Another compute process uses a physical GPU; keep the measurement exclusive\')\n    inventory = command([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.used\',\'--format=csv,noheader,nounits\'])\n    if sum(int(line.rsplit(\',\',1)[1].strip()) for line in inventory.splitlines()) > 1000:\n        raise RuntimeError(\'Physical GPUs not idle; do not overlap work\')\n    a.out_root.mkdir(parents=True,exist_ok=True)\n    job = a.out_root / (\'lora-\'+time.strftime(\'%Y%m%dT%H%M%SZ\',time.gmtime())+\'-\'+uuid.uuid4().hex[:8])\n    job.mkdir(); staged=job/\'repo\'; staged.mkdir()\n    for dirname in (\'harness\',\'task\',\'tests\'):\n        shutil.copytree(a.repo/dirname,staged/dirname,ignore=shutil.ignore_patterns(\'__pycache__\',\'*.pyc\'))\n    for name in (\'README.md\',\'LICENSE\',\'NOTICE\'):\n        if (a.repo/name).is_file(): shutil.copy2(a.repo/name,staged/name)\n    shutil.copytree(a.solution,staged/\'solution\',ignore=shutil.ignore_patterns(\'__pycache__\',\'*.pyc\'))\n    (staged/\'data\').mkdir()\n    for name in DATA: shutil.copy2(a.repo/\'data\'/name,staged/\'data\'/name)\n    # Do not copy adapters, optimizer state, runs, or compiler caches.\n    save(job/\'environment.json\',{\'backend\':a.backend,\'packages\':versions,\'python\':platform.python_version(),\n        \'platform\':platform.platform(),\'cpu_affinity\':sorted(os.sched_getaffinity(0)),\n        \'nvidia_smi\':inventory,\'gpu_probe_short_lived\':gpu_probe,\'gpu_lock\':\'/tmp/soup-gpu0.lock\',\'official_commit\':COMMIT,\'frozen_source_sha256\':frozen,\n        \'hardware_equivalence_claim\':False})\n    save(job/\'assets.json\',assets)\n    save(job/\'source.json\',{str(f.relative_to(staged)):sha(f) for f in sorted((staged/\'solution\').rglob(\'*.py\'))})\n    out=job/\'runs\'\n    env=dict(os.environ)\n    cmd=[sys.executable,str(staged/\'harness/run_task.py\'),\'--repeats\',\'2\',\'--seed\',\'1\',\'--timeout\',\'900\',\'--out\',str(out)]\n    save(job/\'invocation.json\',{\'argv\':cmd,\'baseline_id\':None,\'native_budget_s\':900,\n        \'order\':[\'base\',\'solution\',\'solution\',\'base\'],\'notes\':\'Own same-session baseline; no transplanted Kaggle 79 s budget. Offline warmup reuses cache.\'})\n    print(\'Evidence directory: \'+str(job),flush=True)\n    started=time.perf_counter()\n    with (job/\'harness.log\').open(\'x\') as log:\n        try:\n            code=process_guard.run(cmd,cwd=staged,env=env,stdout=log)\n        except BaseException as exc:\n            save(job/\'assessment.json\',{\'status\':\'FAILED_OR_INTERRUPTED\',\'score\':None,\'admission\':\'UNKNOWN\',\n                 \'cleanup_verified\':not process_guard.descendants(),\'error\':repr(exc)})\n            raise\n    wall=time.perf_counter()-started\n    save(job/\'harness_wall.json\',{\'outer_returncode\':code,\'whole_harness_wall_s\':wall})\n    if code != 0 or not (out/\'result.json\').is_file():\n        save(job/\'assessment.json\',{\'status\':\'FAILED_OR_INCOMPLETE\',\'score\':None,\'admission\':\'UNKNOWN\',\'outer_returncode\':code})\n        raise RuntimeError(\'Harness failed; retained logs, no score/admission\')\n    result=json.loads((out/\'result.json\').read_text())\n    assessment=assess(result,out)\n    # Preserve result.json verbatim even if the native wrapper exits zero on a failure.\n    save(job/\'assessment.json\',assessment)\n    if assessment[\'status\']!=\'COMPLETE_MEASUREMENT\':\n        print(json.dumps(assessment,indent=2)); raise RuntimeError(\'Nested result incomplete; no score/admission\')\n    with (job/\'official-checks.log\').open(\'x\') as log:\n        check_code=process_guard.run([sys.executable,str(staged/\'harness/check_rules.py\'),\'--result\',str(out/\'result.json\')],\n                                     cwd=staged,env=env,stdout=log,timeout=900)\n    assessment.update(official_checks_returncode=check_code, admission=\'UNKNOWN\',score=None,\n                      final_official_kaggle_admission=\'SEPARATE_JURY_GATE\')\n    for rel,expected in frozen.items():\n        if sha(staged/rel)!=expected: raise RuntimeError(\'Frozen file changed during execution: \'+rel)\n    save(job/\'assessment.json\',assessment)\n    print(json.dumps(assessment,indent=2))\n    if check_code: raise RuntimeError(\'Unchanged official checks failed; research evidence only\')\n\n\nif __name__==\'__main__\':\n    try:main()\n    finally:process_guard.finish_quarantine()\n', 'process_guard.py': '"""Linux-only dedicated-runner cleanup; no GPU imports or background activity.\n\nOriginal task-orchestration helper. Not part of solution/ or own-kernel credit.\n"""\nimport ctypes\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport time\n\n\ndef prepare():\n    """Fail before GPU work if race-safe process ownership is unavailable."""\n    if not hasattr(os,\'pidfd_open\') or not hasattr(signal,\'pidfd_send_signal\'):\n        raise RuntimeError(\'Linux pidfd process ownership required\')\n    fd=os.pidfd_open(os.getpid());os.close(fd)\n    libc=ctypes.CDLL(None,use_errno=True)\n    if libc.prctl(36,1,0,0,0)!=0:\n        raise RuntimeError(\'Cannot enable child subreaper; no GPU job started\')\n    enabled=ctypes.c_int(0)\n    if libc.prctl(37,ctypes.byref(enabled),0,0,0)!=0 or enabled.value!=1:\n        raise RuntimeError(\'Child subreaper verification failed\')\n\n\ndef identity(pid):\n    try:\n        fields=Path(f\'/proc/{pid}/stat\').read_text().rsplit(\')\',1)[1].split()\n        return int(fields[1]),int(fields[19]),fields[0]\n    except (OSError,ValueError,IndexError):return None\n\n\ndef descendants():\n    records={}\n    for p in Path(\'/proc\').iterdir():\n        if p.name.isdigit():\n            rec=identity(int(p.name))\n            if rec is not None:records[int(p.name)]=rec\n    owned={os.getpid()}\n    while True:\n        add={pid for pid,(parent,_,_) in records.items() if parent in owned}-owned\n        if not add:break\n        owned.update(add)\n    return {pid:records[pid] for pid in owned if pid!=os.getpid() and pid in records}\n\n\ndef signal_exact(pid,birth,sig):\n    """Never signal a recycled PID. The FD pins identity through the send."""\n    try:fd=os.pidfd_open(pid)\n    except ProcessLookupError:return\n    try:\n        now=identity(pid)\n        if now is not None and now[1]==birth:\n            try:signal.pidfd_send_signal(fd,sig,None,0)\n            except ProcessLookupError:pass\n    finally:os.close(fd)\n\n\ndef _try_signal(pid,birth,sig):\n    try:signal_exact(pid,birth,sig)\n    except OSError as exc:\n        # Do not unwind/release GPU ownership when cleanup is uncertain.\n        quarantine(\'Cleanup signal failed: \'+repr(exc))\n        print(\'SOUP_CLEANUP_PENDING \'+repr(exc),file=sys.stderr,flush=True)\n\n\ndef cleanup(proc):\n    saved={s:signal.signal(s,signal.SIG_IGN) for s in (signal.SIGINT,signal.SIGTERM)}\n    try:\n        frozen=set()\n        while True:\n            observed=descendants()\n            new={(pid,v[1]) for pid,v in observed.items()}-frozen\n            if not new:break\n            for pid,birth in new:_try_signal(pid,birth,signal.SIGSTOP)\n            frozen.update(new)\n        for pid,birth in frozen:_try_signal(pid,birth,signal.SIGKILL)\n        proc.wait()\n        # No timeout-based release: if a GPU child is stuck in kernel exit,\n        # retain this supervisor and its lock until exit is actually observed.\n        while True:\n            while True:\n                try:\n                    pid,_=os.waitpid(-1,os.WNOHANG)\n                    if not pid:break\n                except ChildProcessError:break\n            remaining=descendants()\n            if not remaining:break\n            for pid,(_,birth,_) in remaining.items():_try_signal(pid,birth,signal.SIGKILL)\n            time.sleep(.05)\n    finally:\n        for sig,handler in saved.items():signal.signal(sig,handler)\n\n\ndef run(argv,*,cwd,env,stdout,timeout=None,stderr=subprocess.STDOUT):\n    """Run in dedicated supervisor only; cleanup nested sessions on any failure."""\n    proc=subprocess.Popen(argv,cwd=cwd,env=env,stdout=stdout,stderr=stderr,start_new_session=True)\n    try:\n        code=proc.wait(timeout=timeout)\n        if code < 0:quarantine(\'Child terminated by signal \'+str(-code))\n        if descendants():\n            quarantine(\'Child exited leaving descendants\')\n            cleanup(proc)\n            raise RuntimeError(\'Child exited leaving descendants; cleaned and rejected\')\n        return code\n    except BaseException as exc:\n        quarantine(\'Outer timeout/interruption/error: \'+repr(exc))\n        cleanup(proc)\n        raise\n\n\ndef install_signal_handlers():\n    def interrupted(signum,frame):\n        quarantine(\'Runner interrupted by signal \'+str(signum))\n        raise InterruptedError(\'Runner interrupted by signal \'+str(signum))\n    for sig in (signal.SIGINT,signal.SIGTERM):signal.signal(sig,interrupted)\n\n# A crash-surviving marker is placed before any GPU subprocess. Only its exact\n# owner may clear it on a clean, non-interrupted exit with no live descendants.\n# Interrupted/timed-out work leaves it for explicit operator reconciliation.\n_MARKER = None\n_TOKEN = None\n_KEEP_MARKER = False\n\n\ndef check_quarantine(path):\n    if Path(path).exists():\n        raise RuntimeError(\'GPU0 quarantined: reconcile \'+str(path)+\' before another run\')\n\n\ndef begin_quarantine(path):\n    global _MARKER,_TOKEN,_KEEP_MARKER\n    import json,uuid\n    _MARKER=Path(path);_TOKEN=uuid.uuid4().hex;_KEEP_MARKER=False\n    fd=os.open(str(_MARKER),os.O_WRONLY|os.O_CREAT|os.O_EXCL|os.O_NOFOLLOW,0o600)\n    with os.fdopen(fd,\'w\') as out:\n        json.dump({\'owner_token\':_TOKEN,\'owner_pid\':os.getpid(),\'status\':\'OWNED_GPU_WORK_IN_PROGRESS\'},out)\n        out.flush();os.fsync(out.fileno())\n\n\ndef quarantine(reason):\n    global _KEEP_MARKER\n    _KEEP_MARKER=True\n    if _MARKER is None:return\n    import json\n    try:value=json.loads(_MARKER.read_text())\n    except (OSError,ValueError):return\n    if value.get(\'owner_token\')!=_TOKEN:return\n    value.update(status=\'BLOCKED_OPERATOR_RECONCILIATION_REQUIRED\',reason=str(reason))\n    _MARKER.write_text(json.dumps(value)+\'\\n\')\n\n\ndef finish_quarantine():\n    if _MARKER is None or _KEEP_MARKER:return\n    if descendants():\n        quarantine(\'Live owned descendants remain at supervisor exit\')\n        return\n    import json\n    try:value=json.loads(_MARKER.read_text())\n    except (OSError,ValueError):return\n    if value.get(\'owner_token\')==_TOKEN:_MARKER.unlink()\n'}
EXPECTED = {'solution/__init__.py': 'dcf225b5c2da76e6680cc66bcda674699ce6c33f35d3c655498dee39333d505a', 'solution/epilogue.py': '04e1a6dd75fa8dc9fba42123a6bd6dd224649791e782200927328eba8630a7ec', 'solution/patch.py': '3d5920504c3c46433e82c93be3b0ac305274ca982679b3c3f80757dfb57e6840', 'run_prepared.py': '7dafd5d5c4c709e2dcf0271888cc30332c23ce91990c17a0b744cf4eec803cbe', 'process_guard.py': 'f223349a54ab7bda32de1bf0f28aa86ff5c656d2939c74acec1ea49cd4aaa4ff'}
for name, source in PAYLOAD.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED[name], name
    destination = PACKAGE / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source)
print('Embedded source hashes verified:', len(PAYLOAD))


## Unchanged official paired run

Uses **base, solution, solution, base**, seed 1, native timeout 900 s per run.
This records an independent baseline in this same environment: GCP is not Kaggle.
Full native command wall, external nvidia-smi peak, quality and useful own-kernel
share (steps 3 onward) are read from unchanged official evidence. A separate outer
harness duration is recorded. No historical 79 s limit is transplanted.

The runner retains logs and raw results, rejects partial/failed native runs even if
wrapper exit is zero, and only then invokes unchanged official checks/regressions.
`score` remains null and `admission` UNKNOWN. Hidden-quality/final Kaggle admission
are separate. Flushed kernel launch-attempt/submitted markers survive train-log timeouts;
submitted is **not** synchronized GPU completion.


In [ ]:
cmd = [sys.executable, str(PACKAGE / 'run_prepared.py'),
       '--repo', str(OFFICIAL_REPO), '--cache', str(HF_CACHE),
       '--out-root', str(RUN_ROOT), '--backend', BACKEND]
# The native harness supplies finite per-process timeouts and cleanup. Do not run
# concurrent jobs on this GPU or interrupt and immediately start another job.
import signal
if not AUTHORIZE_GPU_EXECUTION:
    raise RuntimeError('Prepared only: configure existing cloud paths and explicitly authorize the GPU run first')
proc = subprocess.Popen(cmd, start_new_session=True)
try:
    result = proc.wait()
except KeyboardInterrupt:
    # Ask the supervisor to clean up; do not kill it or release its GPU lock.
    proc.send_signal(signal.SIGTERM)
    while True:
        try:
            proc.wait()
            break
        except KeyboardInterrupt:
            continue
    raise
if result:
    raise subprocess.CalledProcessError(result, cmd)


## Evidence interpretation

Read the emitted evidence directory's `assessment.json`, `harness_wall.json`,
`runs/result.json`, `runs/checks.json`, train logs, source/environment/asset hashes.
A share failure is an actual limitation of this narrow ablation; never pad work to meet it.
The worker accepts reviewed `candidate.zip` solution source, not this notebook.
Do not submit the template until a real measured backend-specific baseline ID is known.
